# ERSP Analysis
Raw signals are loaded from network storage across three patient groups using a single format-agnostic loader that handles TRC, EDF, and H5 files (load_first_raw_in_dir). Non-neural channels are removed (filter_aux_channels), and for EL patients only electrodes with anatomical labels are kept. Signals are then rereferenced to the average of white matter contacts defined per patient (apply_wm_reref), and line noise is removed at each harmonic only if a real peak is detected, with notch strength set automatically (notch_mains_harmonics). Trial timing comes from photodiode triggers saved as TSV files per patient, and trials are kept only if the stimulus lasted at least 0.5s and the response no more than 10s, with IQR used to remove remaining outliers (collect_trials). ERSPs are computed using short-time Fourier transform and warped so each trial is split 50/50 between stimulus and post-stimulus, baseline corrected before stimulus onset (compute_ersp). White matter channels are skipped. Outputs per channel are an ERSP plot, a high-gamma heatmap sorted by trial duration (plot_hg_trials), and for clustering a raw matrix and a clean image. QC outputs are two PSDs (before and after processing) and a full recording montage with trial markers (plot_montage_overview).

For each patient, loads and preprocesses raw neural signals, then runs one or both of two parallel pipelines controlled by boolean flags.

## Processing Steps (shared for all patients)

#### 1. Data Loading
- Builds patient-specific paths (raw + prep directories)
- Loads raw signals using format-agnostic loader (TRC/EDF/H5)
- Removes auxiliary channels (ECG, DC, markers etc.)

#### 2. Channel Filtering (EL patients only)
- **SEEG patients**: keeps only channels with `_` in name (e.g. `A_L6`)
- **Grid patients** (e.g. EL044): keeps only channels matching defined prefixes with a digit (e.g. `Pa1`, `T17`, `postP3`)
- Skips patient entirely if no neural channels remain

#### 3. Preprocessing
- Saves **PSD before processing** (if flag on)
- Applies **white matter rereferencing**
- Applies **adaptive mains notch filtering**
- Saves **PSD after processing** (if flag on)

#### 4. Trial Collection
- Reads trial TSV files from `prep0`
- Applies hard duration filters: `min_stim_s=0.5`, `max_post_s=10`
- Trims outliers using IQR method
- Saves QC report and histogram


## Pipeline A — ERSP Pipeline
*Runs if `RUN_ERSP_PIPELINE=True`*

- Saves **montage overview plot** with trial onset/offset markers
- For each condition and channel:
  - Computes **ERSP** (time-frequency power map)
  - Saves **ERSP plot** (`.tif`)
  - Saves **HG trials plot** (high-gamma, trial-by-trial heatmap)

## Pipeline B — Cluster Export
*Runs if `RUN_CLUSTER_EXPORT=True`*

- Skips non-neural, bad, and WM channels
- For each condition and channel:
  - Reuses ERSP result if Pipeline A also ran (no recomputation)
  - Saves **ERSP matrix** (`.npy`) for clustering input
  - Saves **clean ERSP image** (`.png`) for clustering input

## Outputs
| Product | Location | Pipeline |
|---|---|---|
| PSD_raw | `outputs/04_ersp_LM/<pid>/LM/PSD_raw` | A |
| PSD_clean | `outputs/04_ersp_LM/<pid>/LM/PSD_clean` | A |
| Report + montage | `outputs/04_ersp_LM/<pid>/LM/Report` | A |
| ERSP plots | `outputs/04_ersp_LM/<pid>/LM/ERSP/<cond>` | A |
| HG plots | `outputs/04_ersp_LM/<pid>/LM/HG/<cond>` | A |
| ERSP matrix | `outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>` | B |
| ERSP clean PNG | `outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_clean/<cond>` | B |

## Imports & run controls 
(the only place you change things is here and cell 4)


In [ ]:
# ============================================================
# 140_ERSP_analysis_pipeline.ipynb
# Cell 1 — Imports & run controls
# ============================================================
import os, glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from itertools import groupby
import csv

from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_PATIENTS, MICROEPI_PRESETS, COND_ALIAS)
import LFfunctions_PDextract as LF
from LF_pd import load_patient_raw

# ------------------------------------------------------------------
# RUN CONTROLS  ← edit these before each run
# ------------------------------------------------------------------
BLOCK = "LM"

# Toggle sections
RUN_PD_EXTRACTION    = False
RUN_ERSP_PIPELINE    = False
RUN_CLUSTER_EXPORT   = False
DO_MONTAGE_PSD_PLOTS = True

# Output roots
ERSP_SCRIPT_NAME    = "04_ersp_LM"
RAWONLY_SCRIPT_NAME = "04_ersp_LM_RAWONLY"
run_root_ersp       = os.path.join(cfg.outputs_root, ERSP_SCRIPT_NAME)
run_root_raw        = os.path.join(cfg.outputs_root, RAWONLY_SCRIPT_NAME)

# ERSP params (from config.py)
ersp_params = fe.ERSPParams(
    nperseg=cfg.nperseg, nfft=cfg.nfft, noverlap=cfg.noverlap,
    baseline_w=cfg.baseline_w, proportions=cfg.proportions,
    n_time_bins=cfg.n_time_bins, vmin=cfg.vmin, vmax=cfg.vmax, fmax=cfg.fmax
)

print("Controls loaded.")

In [ ]:
# ============================================================
# Cell 2 — Helper functions
# (implementations live in lf_ersp.py and lf_io_utils.py)
# ============================================================

# Direct aliases
notch_mains_harmonics = fe.notch_mains_harmonics
fill_nans_nearest     = fe.fill_nans_nearest
save_clean_png        = fe.save_clean_png
plot_psd_overview     = fe.plot_psd_overview
_is_non_neural        = io._is_non_neural
_ensure               = io.ensure_dir

# Thin cfg-binding wrappers (keep pipeline cells unchanged)
def apply_notch_with_audit(signals, fs, patient_id, pid_raw):
    return fe.apply_notch_with_audit(
        signals, fs, patient_id, pid_raw,
        notch_patients=getattr(cfg, "notch_patients", []),
        mains_base=getattr(cfg, "mains_base", 50.0),
        fmax=getattr(cfg, "fmax", 500.0),
        repeats=getattr(cfg, "notch_repeats", 1),
    )

def apply_wm_reref(signals, names, patient_id):
    """Returns (signals, reref_label, wm_skip_set)."""
    if str(cfg.reref_type).upper() != "WM":
        return signals, "None", set()
    wm_idx = io.wm_indices_for_patient(patient_id, names)
    if not wm_idx:
        return signals, "None", set()
    bad = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
    signals_r, used, excluded = fe.apply_wm_reference_with_exclusions(
        signals, names, wm_idx, bad)
    reref = "WM" if used else "None"
    return signals_r, reref, set(used) | set(excluded)

print("Helpers loaded.")

## Part 1 — Photodiode / trial extraction
Runs `LF_pd.load_patient_raw` and `LFfunctions_PDextract` for each patient in `PD_PATIENTS`.
Output: TSV timing files saved to each patient's `prep0` folder.
Set `RUN_PD_EXTRACTION = False` in Cell 1 to skip.

### PD extraction loop (PAT / EL / MicroEPI unified, calls LF_pd as-is)


In [ ]:
if not RUN_PD_EXTRACTION:
    print("[skip] PD extraction (RUN_PD_EXTRACTION=False)")
else:
    # Active patients (empty list = skip that group)
    PAT_PATIENTS      = []
    EL_PATIENTS       = ["EL042","EL043","EL044","EL045"]
    MICROEPI_PATIENTS = []

    all_patients = (
        [(pid, "PAT",      PAT_PRESETS)      for pid in PAT_PATIENTS] +
        [(pid, "EL",       EL_PRESETS)       for pid in EL_PATIENTS] +
        [(pid, "MICROEPI", MICROEPI_PRESETS) for pid in MICROEPI_PATIENTS]
    )

    for pid, group, presets in all_patients:
        preset = presets.get(pid)
        if preset is None:
            print(f"[skip] {pid}: no preset"); continue

        patient_id = f"PAT_{pid}" if group == "PAT" else f"MicroEPI-{pid}" if group == "MICROEPI" else pid
        print(f"\n=== {patient_id} ===")

        try:
            if group == "PAT":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\{patient_id}\task_FBM\data_{BLOCK}\raw"
                raw_signals, channel_names, sampling_rate = io.load_trc_and_signals(glob.glob(os.path.join(base_path, "*.TRC"))[0])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            elif group == "EL":
                info          = load_patient_raw(pid, block_name=BLOCK, use_el_mat_fallback=False, verbose=True)
                raw_signals   = info["raw_signals"]
                sampling_rate = info["sampling_rate"]
                channel_names = list(info["channel_names"])
                save_path     = info["save_path"]
                exp_file      = info["matching_files_onsets"][0] if info["matching_files_onsets"] else None

            elif group == "MICROEPI":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\{patient_id}\task_FBM\data_{BLOCK}\raw"
                buckets   = _load_microepi_folder(base_path)
                if not buckets: raise RuntimeError("No .mat files found")
                trig_key  = preset["trig"].lower()
                kind      = next((k for k in ('micro','macro') if k in buckets and any(c.lower()==trig_key for c in buckets[k][2])), list(buckets.keys())[0])
                raw_signals, sampling_rate, channel_names = buckets[kind][0], buckets[kind][1], list(buckets[kind][2])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            if group != "EL":
                exp_files = glob.glob(os.path.join(base_path, "*.tsv")) or glob.glob(os.path.join(base_path, "*.txt"))
                exp_file  = exp_files[0] if exp_files else None

            lower_map = {str(c).lower(): str(c) for c in channel_names}
            trig_key  = preset["trig"].lower()
            if trig_key not in lower_map:
                print(f"  [warn] trigger '{preset['trig']}' not found — skipping"); continue
            pd_name = lower_map[trig_key]

            mt = preset["manual_trig"]
            manual_path = None
            if mt:
                manual_path = mt if os.path.isabs(mt) else os.path.join(save_path, mt)
                if not os.path.exists(manual_path):
                    print(f"  [warn] manual triggers not found: {manual_path}")
                    manual_path = None

            on_abs, off_abs, metrics = LF.get_trigger_indexes_photodiode(
                raw_signals=raw_signals, sampling_rate=sampling_rate,
                channel_names=channel_names, trig_name=pd_name,
                time_range=preset["time_range"], threshold_val=0.40,
                flip_trigs=preset["flip"],
                trial_ids=preset["trial_ids"], invalid_trials=preset["invalid_trials"],
                ignore_invalid=False, fake_trials=preset["fake_trials"],
                extra_table_path=exp_file, manual_trigs_path=manual_path,
                return_extra_metrics=True,
            )
            print(f"  Paired trials: {len(on_abs)}")
            LF.parse_and_save(pid, patient_id, on_abs, off_abs, metrics,
                              sampling_rate, save_path, BLOCK, exp_file,
                              preset["trial_ids"], preset["trig"],
                              cond_alias=COND_ALIAS)

        except Exception as e:
            print(f"[error] {patient_id}: {e}")

    print("\n[PD extraction done]")

## Part 2 — ERSP + HG + Report pipeline
Reads raw data and `prep0` TSVs. Outputs:
- `04_ersp_LM/` → ERSP plots, HG plots, ERSP_significant, Report TSVs
Set `RUN_ERSP_PIPELINE = False` in Cell 1 to skip.

In [ ]:
if not RUN_ERSP_PIPELINE:
    print("[skip] ERSP pipeline (RUN_ERSP_PIPELINE=False)")
else:
    for pid_raw in cfg.patient_ids:
        try:
            patient_id, raw_dir, prep_dir = io.build_paths_for_patient(pid_raw, cfg.block_name)
            signals, names, fs = io.load_first_raw_in_dir(raw_dir)
            signals, names, *_ = io.filter_aux_channels(signals, names)

            if str(pid_raw).startswith("EL"):
                if pid_raw in cfg.EL_GRID_PATIENTS:
                    prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, ())
                    keep = [i for i, nm in enumerate(names)
                            if any(str(nm).startswith(p) for p in prefixes)
                            and any(c.isdigit() for c in str(nm))]
                else:
                    keep = [i for i, nm in enumerate(names) if "_" in str(nm)]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                if len(names) == 0:
                    print(f"[skip] {patient_id}: no neural channels"); continue

            if DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_raw"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600
                )

            signals, reref, wm_skip = apply_wm_reref(signals, names, patient_id)
            signals = apply_notch_with_audit(signals, fs, patient_id, pid_raw)

            if DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_clean"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600
                )

            report_dir  = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "Report")
            report_path = os.path.join(report_dir, f"{patient_id}_IQR.tsv")
            cond_groups = tr.collect_trials(prep_dir, fs, outlier_method="IQR",
                                            iqr_k=cfg.iqr_k, report_path=report_path,
                                            patient_id=patient_id, max_post_s=cfg.max_post_s)
            if not cond_groups:
                print(f"[skip] {patient_id}: no trials"); continue

            tr.plot_montage_overview(
                signals=signals, fs=fs, names=names,
                cond_groups=cond_groups, save_dir=report_dir, patient_id=patient_id,
            )

            ersp_root = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "ERSP")
            hg_root   = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "HG")

            for cond, (onsets, offsets, trial_ends) in cond_groups.items():
                ersp_dir = _ensure(os.path.join(ersp_root, cond))
                hg_dir   = _ensure(os.path.join(hg_root, cond))
                for ci, chan_name in enumerate(names):
                    if chan_name in wm_skip: continue
                    res = fe.compute_ersp(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        trial_ends=trial_ends, mode=cfg.mode, time_window=cfg.time_window,
                        params=ersp_params
                    )
                    fe.plot_ersp(res, patient_id=patient_id, condition=cond,
                                 reref_type=reref, chan_name=chan_name,
                                 save_dir=ersp_dir, params=ersp_params,
                                 plot_title=False, save_sidecar=False)
                    fe.plot_hg_trials(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        chan_name=chan_name, patient_id=patient_id, condition=cond, reref_type=reref,
                        time_window=cfg.time_window, baseline_w=cfg.baseline_w,
                        hg_band=cfg.hg_band, smooth_ms=cfg.hg_smooth_ms,
                        vmin=cfg.hg_vmin, vmax=cfg.hg_vmax,
                        save_dir=hg_dir, add_separators=False, sort_ascending=True,
                        trial_end_indices=trial_ends, sort_by="stim"
                    )
            print(f"[done] {pid_raw}")

        except Exception as e:
            print(f"[error] {pid_raw}: {e}")

    print("\nERSP pipeline done. Outputs:", run_root_ersp)

## Part 3 — Clustering export (ERSP_matrix + ERSP_clean)
Reads raw data and `prep0` TSVs. Outputs:
- `04_ersp_LM_RAWONLY/` → ERSP_matrix (.npy), ERSP_clean (.png), Report TSV
Set `RUN_CLUSTER_EXPORT = False` in Cell 1 to skip.

In [ ]:
if not RUN_ERSP_PIPELINE and not RUN_CLUSTER_EXPORT:
    print("[skip] both pipelines disabled")
else:
    for pid_raw in cfg.patient_ids:
        try:
            patient_id, raw_dir, prep_dir = io.build_paths_for_patient(pid_raw, cfg.block_name)
            signals, names, fs = io.load_first_raw_in_dir(raw_dir)
            signals, names, *_ = io.filter_aux_channels(signals, names)

            if str(pid_raw).startswith("EL"):
                if pid_raw in cfg.EL_GRID_PATIENTS:
                    prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, ())
                    keep = [i for i, nm in enumerate(names)
                            if any(str(nm).startswith(p) for p in prefixes)
                            and any(c.isdigit() for c in str(nm))]
                else:
                    keep = [i for i, nm in enumerate(names) if "_" in str(nm)]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                if len(names) == 0:
                    print(f"[skip] {patient_id}: no neural channels"); continue

            if RUN_ERSP_PIPELINE and DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_raw"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600
                )

            signals, reref, wm_skip = apply_wm_reref(signals, names, patient_id)
            signals = apply_notch_with_audit(signals, fs, patient_id, pid_raw)

            if RUN_ERSP_PIPELINE and DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_clean"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600
                )

            report_dir  = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "Report")
            report_path = os.path.join(report_dir, f"{patient_id}_IQR.tsv")
            cond_groups = tr.collect_trials(prep_dir, fs, outlier_method="IQR",
                                            iqr_k=cfg.iqr_k, report_path=report_path,
                                            patient_id=patient_id, max_post_s=cfg.max_post_s)
            if not cond_groups:
                print(f"[skip] {patient_id}: no trials"); continue

            if RUN_ERSP_PIPELINE:
                tr.plot_montage_overview(
                    signals=signals, fs=fs, names=names,
                    cond_groups=cond_groups, save_dir=report_dir, patient_id=patient_id,
                )
                ersp_root = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "ERSP")
                hg_root   = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "HG")

            if RUN_CLUSTER_EXPORT:
                mat_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_matrix"))
                img_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_clean"))
                skip     = set(n for n in names if _is_non_neural(n))
                skip    |= set(getattr(cfg, "bad_channels_manual", {}).get(patient_id, []))
                skip    |= wm_skip

            for cond, (onsets, offsets, trial_ends) in cond_groups.items():
                if RUN_ERSP_PIPELINE:
                    ersp_dir = _ensure(os.path.join(ersp_root, cond))
                    hg_dir   = _ensure(os.path.join(hg_root, cond))
                if RUN_CLUSTER_EXPORT:
                    out_mat  = _ensure(os.path.join(mat_root, cond))
                    out_png  = _ensure(os.path.join(img_root, cond))

                for ci, chan_name in enumerate(names):
                    if chan_name in wm_skip: continue
                    if RUN_CLUSTER_EXPORT and chan_name in skip and not RUN_ERSP_PIPELINE: continue

                    res = fe.compute_ersp(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        trial_ends=trial_ends, mode=cfg.mode, time_window=cfg.time_window,
                        params=ersp_params
                    )

                    if RUN_ERSP_PIPELINE:
                        fe.plot_ersp(res, patient_id=patient_id, condition=cond,
                                     reref_type=reref, chan_name=chan_name,
                                     save_dir=ersp_dir, params=ersp_params,
                                     plot_title=False, save_sidecar=False)
                        fe.plot_hg_trials(
                            signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                            chan_name=chan_name, patient_id=patient_id, condition=cond, reref_type=reref,
                            time_window=cfg.time_window, baseline_w=cfg.baseline_w,
                            hg_band=cfg.hg_band, smooth_ms=cfg.hg_smooth_ms,
                            vmin=cfg.hg_vmin, vmax=cfg.hg_vmax,
                            save_dir=hg_dir, add_separators=False, sort_ascending=True,
                            trial_end_indices=trial_ends, sort_by="stim"
                        )

                    if RUN_CLUSTER_EXPORT and chan_name not in skip:
                        A = np.array(res["avg_db"], float)
                        if np.isnan(A).any():
                            print(f"[warn] {patient_id} {cond} {chan_name}: {int(np.isnan(A).sum())} NaNs → filling")
                            fill_nans_nearest(A)
                        mode_tag = "_TN" if str(res["meta"]["mode"]).upper() == "TN" else ""
                        stem = f"{patient_id}_{cond}_{reref}_ERSP_{chan_name}{mode_tag}"
                        np.save(os.path.join(out_mat, f"{stem}.npy"), A)
                        save_clean_png(A, vmin=ersp_params.vmin, vmax=ersp_params.vmax,
                                       path_png=os.path.join(out_png, f"{stem}_CLEAN.png"))

                print(f"[done] {patient_id} – {cond}")

        except Exception as e:
            print(f"[error] {pid_raw}: {e}")

    print("\nPipeline done.")

## Cell 7 — Clustering export section header (markdown)


## Cell 8 — ERSP matrix + clean PNG export loop (04_ersp_LM_RAWONLY outputs)